# Lagrangian Frame Frequency Spectra

Trace a trajectory following the local E×B drift velocity and compare
the frequency PSD measured in the co-moving (Lagrangian) frame with
the lab-frame (Eulerian) PSD at a fixed point.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from reconn_wave_power.io import read_simulation
from reconn_wave_power.spectrum import compute_psd_time
from reconn_wave_power.lagrangian import (
    compute_exb_velocity,
    trace_trajectory,
    sample_along_trajectory,
    lagrangian_psd,
)

%matplotlib inline

## Load Data

In [ ]:
INPUT_FILE = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/input/input"
OUTPUT_FOLDER = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/Output"

ds = read_simulation(
    input_file=INPUT_FILE,
    output_folder=OUTPUT_FOLDER,
    fields=("E", "B"),
    progress=True,
)
ds

In [ ]:
dx = float(ds.x[1] - ds.x[0])
dy = float(ds.y[1] - ds.y[0])
dt = ds.attrs.get("dt", float(ds.time[1] - ds.time[0]))

print(f"dx = {dx}, dy = {dy}, dt = {dt}")
print(f"x range: [{float(ds.x.min())}, {float(ds.x.max())}]")
print(f"y range: [{float(ds.y.min())}, {float(ds.y.max())}]")
print(f"time range: [{float(ds.time.min())}, {float(ds.time.max())}]")
print(f"timesteps: {len(ds.time)}")

## E×B Drift Velocity

Visualise the in-plane E×B drift at a representative timestep.

In [ ]:
SNAPSHOT_IDX = len(ds.time) // 2

vx, vy = compute_exb_velocity(ds, time_idx=SNAPSHOT_IDX)

fig, axes = plt.subplots(2, 1, figsize=(7, 5))

for ax, v, label in zip(axes, [vx, vy], [r"$v_{E\times B,x}$", r"$v_{E\times B,y}$"]):
    im = ax.pcolormesh(ds.x, ds.y, v.values.T, shading="auto", cmap="RdBu_r")
    ax.set_xlabel(r"x [$d_i$]")
    ax.set_ylabel(r"y [$d_i$]")
    ax.set_title(label)
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax)

fig.suptitle(f"E×B drift at t = {float(ds.time[SNAPSHOT_IDX]):.1f}", fontsize=14)
plt.tight_layout()
plt.show()

## Configuration

Choose the starting point, field component, and PSD parameters.

In [ ]:
FIELD = "Bz"

# Starting point for the Lagrangian trajectory
X0 = float(ds.x[len(ds.x) // 4])   # quarter of the way through
Y0 = float(ds.y[len(ds.y) // 2])   # centre of domain
T0_IDX = len(ds.time) // 2          # start in the middle of the run

print(f"Starting point: x0 = {X0:.1f}, y0 = {Y0:.1f}")
print(f"Starting timestep index: {T0_IDX} (t = {float(ds.time[T0_IDX]):.1f})")

## Trace Trajectory

In [ ]:
x_traj, y_traj, t_traj = trace_trajectory(ds, X0, Y0, T0_IDX, progress=True)

print(f"Trajectory length: {len(t_traj)} points")
print(f"Time span: [{t_traj[0]:.1f}, {t_traj[-1]:.1f}]")
print(f"x span: [{x_traj.min():.1f}, {x_traj.max():.1f}]")
print(f"y span: [{y_traj.min():.1f}, {y_traj.max():.1f}]")

### Plot trajectory over the 2D domain

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

snapshot = ds[FIELD].isel(time=T0_IDX)
im = ax.pcolormesh(ds.x, ds.y, snapshot.values.T, shading="auto", cmap="RdBu_r")
ax.plot(x_traj, y_traj, "k-", lw=1.5, label="E×B trajectory")
ax.plot(X0, Y0, "ko", ms=8, label="start")
ax.set_xlabel(r"x [$d_i$]")
ax.set_ylabel(r"y [$d_i$]")
ax.set_title(f"{FIELD} at t = {float(ds.time[T0_IDX]):.1f} with Lagrangian trajectory")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.colorbar(im, ax=ax, label=FIELD)
plt.tight_layout()
plt.show()

In [ ]:
x_traj, y_traj

### Trajectory in x-t space

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# x-t slice at the starting y
da_xt = ds[FIELD].sel(y=Y0, method="nearest")
im = ax.pcolormesh(ds.x, ds.time, da_xt.values, shading="auto", cmap="RdBu_r")
ax.plot(x_traj, t_traj, "k-", lw=1.5, label="E×B trajectory")
ax.plot(X0, float(ds.time[T0_IDX]), "ko", ms=8)
ax.set_xlabel(r"x [$d_i$]")
ax.set_ylabel(r"time [$\Omega_{ci}^{-1}$]")
ax.set_title(f"{FIELD}(x, t) at y = {Y0:.1f} with trajectory")
ax.legend(loc="upper right")
plt.colorbar(im, ax=ax, label=FIELD)
plt.tight_layout()
plt.show()

## Extract Field Along Trajectory

In [ ]:
ts_lagrangian = sample_along_trajectory(ds, FIELD, x_traj, y_traj, t_traj)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_traj, ts_lagrangian.values, label="Lagrangian")

# Eulerian time series at the fixed starting point for comparison
ts_eulerian = ds[FIELD].sel(x=X0, y=Y0, method="nearest")
ax.plot(ds.time, ts_eulerian.values, alpha=0.6, label=f"Eulerian (x={X0:.0f}, y={Y0:.0f})")

ax.set_xlabel(r"time [$\Omega_{ci}^{-1}$]")
ax.set_ylabel(FIELD)
ax.set_title(f"{FIELD} time series: Lagrangian vs Eulerian")
ax.legend()
plt.tight_layout()
plt.show()

## Compare Eulerian vs Lagrangian PSD

In [ ]:
# Lagrangian PSD
f_lag, Pxx_lag = compute_psd_time(ts_lagrangian, dt=dt)

# Eulerian PSD (full time range at fixed point)
f_eul, Pxx_eul = compute_psd_time(ts_eulerian, dt=dt)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(f_eul, Pxx_eul, label="Eulerian (fixed point)", alpha=0.8)
ax.semilogy(f_lag, Pxx_lag, label="Lagrangian (E×B frame)", alpha=0.8)
ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
ax.set_ylabel("PSD")
ax.set_title(f"Frequency PSD of {FIELD}: Eulerian vs Lagrangian")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Convenience Wrapper: `lagrangian_psd`

In [ ]:
f, Pxx, traj = lagrangian_psd(ds, FIELD, X0, Y0, T0_IDX, dt)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(f, Pxx, label="Lagrangian PSD (via wrapper)")
ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
ax.set_ylabel("PSD")
ax.set_title(f"Lagrangian PSD of {FIELD}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()